In [ ]:
import torch
import torch.nn.functional as F
import time

device = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch version:", torch.__version__)
print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu130
Device: cuda
GPU: Tesla T4


In [ ]:
import math

def manual_attention(query, key, value):
    scores = torch.matmul(
        query, key.transpose(-2, -1)
    )

    scores = scores / math.sqrt(query.size(-1))

    weights = torch.softmax(scores, dim=-1)

    output = torch.matmul(weights, value)

    return output

In [ ]:
def sdpa_attention(query, key, value):
    return F.scaled_dot_product_attention(
        query,
        key,
        value,
        dropout_p=0.0
    )

In [ ]:
torch.manual_seed(42)

batch_size = 2
num_heads = 4
seq_len = 128
head_dim = 64

query = torch.randn(
    batch_size, num_heads, seq_len, head_dim,
    device=device
)

key = torch.randn_like(query)
value = torch.randn_like(query)

# Without SDPA
output_manual = manual_attention(query, key, value)

# With SDPA
output_sdpa = sdpa_attention(query, key, value)

# Compare the results
max_difference = (
    output_manual - output_sdpa
).abs().max().item()

print("Manual output shape:", output_manual.shape)
print("SDPA output shape:", output_sdpa.shape)
print("Maximum absolute difference:", max_difference)

print(
    "Outputs close:",
    torch.allclose(
        output_manual,
        output_sdpa,
        atol=1e-5,
        rtol=1e-5
    )
)

Manual output shape: torch.Size([2, 4, 128, 64])
SDPA output shape: torch.Size([2, 4, 128, 64])
Maximum absolute difference: 5.960464477539062e-07
Outputs close: True


In [ ]:
batch_size = 2
num_heads = 8
seq_len = 8000
head_dim = 64

query = torch.randn(
    batch_size, num_heads, seq_len, head_dim,
    device=device
)

key = torch.randn_like(query)
value = torch.randn_like(query)

print("Query shape:", query.shape)
print("Sequence length:", seq_len)
print("Device:", device)

Query shape: torch.Size([2, 8, 8000, 64])
Sequence length: 8000
Device: cuda


In [ ]:
def benchmark(fn, warmup=10, iterations=30):
    # Warmup: allow initial setup and kernel selection
    for _ in range(warmup):
        fn()

    if device == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    for _ in range(iterations):
        fn()

    if device == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    return elapsed * 1000 / iterations


manual_ms = benchmark(
    lambda: manual_attention(query, key, value)
)

sdpa_ms = benchmark(
    lambda: sdpa_attention(query, key, value)
)

print(f"Manual attention: {manual_ms:.3f} ms")
print(f"SDPA attention:   {sdpa_ms:.3f} ms")

print(
    f"Speedup: {manual_ms / sdpa_ms:.2f}x"
)

Manual attention: 162.410 ms
SDPA attention:   84.444 ms
Speedup: 1.92x


In [ ]:
def measure_peak_memory(fn):
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    baseline = torch.cuda.memory_allocated()

    output = fn()

    torch.cuda.synchronize()

    peak = torch.cuda.max_memory_allocated()

    peak_extra_mb = (peak - baseline) / (1024 ** 2)

    del output

    return peak_extra_mb


if device == "cuda":
    manual_memory = measure_peak_memory(
        lambda: manual_attention(query, key, value)
    )

    sdpa_memory = measure_peak_memory(
        lambda: sdpa_attention(query, key, value)
    )

    print(f"Manual peak additional memory: {manual_memory:.2f} MiB")
    print(f"SDPA peak additional memory:   {sdpa_memory:.2f} MiB")
else:
    print("Switch to a CUDA GPU runtime to run this experiment.")

Manual peak additional memory: 7844.50 MiB
SDPA peak additional memory:   32.00 MiB


### GPT

In [ ]:
def manual_causal_attention(query, key, value):
    seq_len = query.size(-2)

    scores = torch.matmul(
        query, key.transpose(-2, -1)
    ) / math.sqrt(query.size(-1))

    # True means future positions are blocked
    causal_mask = torch.triu(
        torch.ones(
            seq_len, seq_len,
            device=query.device,
            dtype=torch.bool
        ),
        diagonal=1
    )

    scores = scores.masked_fill(
        causal_mask, float("-inf")
    )

    weights = torch.softmax(scores, dim=-1)

    return torch.matmul(weights, value)

In [ ]:
def sdpa_causal_attention(query, key, value):
    return F.scaled_dot_product_attention(
        query,
        key,
        value,
        is_causal=True,
        dropout_p=0.0
    )

In [ ]:
# Use a smaller sequence for this demonstration
query_small = query[:, :, :128, :]
key_small = key[:, :, :128, :]
value_small = value[:, :, :128, :]

manual_output = manual_causal_attention(
    query_small, key_small, value_small
)

sdpa_output = sdpa_causal_attention(
    query_small, key_small, value_small
)

print(
    "Outputs close:",
    torch.allclose(
        manual_output,
        sdpa_output,
        atol=1e-5,
        rtol=1e-5
    )
)

Outputs close: True
